# IncidentCommander — GRPO training on Colab (HF TRL)

This notebook trains a small LLM (default Qwen2.5-1.5B-Instruct) with GRPO against the live IncidentCommander OpenEnv server on HuggingFace Spaces.

Follows the same pipeline that won the previous OpenEnv Hackathon (kube-sre-gym).

**Runtime**: A100 40GB is fine for Qwen2.5-1.5B with LoRA (r=16). For Qwen3-1.7B use A100 80GB or L4.

In [ ]:
# 1. Install training deps
!pip install -q 'trl>=0.29.0' 'transformers>=4.45' 'peft>=0.13' 'accelerate>=1.0' 'datasets>=2.20' 'vllm>=0.6' 'httpx>=0.27'

In [ ]:
# 2. HuggingFace login (for model downloads + checkpoint push)
from huggingface_hub import login
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
login(os.environ['HF_TOKEN'])

In [ ]:
# 3. Point to our live OpenEnv server on HF Spaces.
#    (Swap in your local http://localhost:7860 if you run the server locally.)
ENV_URL = 'https://sagnik-mukherjee-incodent-commander.hf.space'

import httpx
print(httpx.get(f'{ENV_URL}/health').json())
print(httpx.get(f'{ENV_URL}/curriculum').json())

In [ ]:
# 4. Pull the training script out of the repo so we can import it.
!git clone https://huggingface.co/spaces/sagnik-mukherjee/incodent-commander ic-space
%cd ic-space/rl-agent
!pip install -q -e .

In [ ]:
# 5. Smoke test the rollout loop (no GPU needed).
from training.train_grpo import OpenEnvClient, rollout_episode, episode_reward
import json

client = OpenEnvClient(ENV_URL, persona='senior')

def stub_generate(prompt: str) -> str:
    # Pretend model always inspects payments-api logs first.
    return json.dumps({'action_type': 'query_logs', 'params': {'service': 'payments-api', 'last_minutes': 5}})

r = rollout_episode(client, stub_generate, use_curriculum=True, adversarial=False, step_limit=6)
print('task:', r.task_id, 'tier:', r.tier)
print('cum reward:', r.cumulative_reward)
print('grade:', r.grade)
print('GRPO reward:', episode_reward(r))

In [ ]:
# 6. Actual GRPO training run (REQUIRES GPU)
# With HF college credits, set HUB_REPO to <your-org>/incident-commander-grpo
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.train_grpo',
    '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--env-url', ENV_URL,
    '--num-generations', '8',
    '--max-steps', '200',
    '--save-steps', '20',
    '--grad-accum', '8',
    '--lr', '1e-5',
    '--hub-repo', 'your-name/incident-commander-grpo',  # change this
    '--vllm-mode', 'colocate',
]
print(' '.join(cmd))
# Uncomment when GPU is available:
# subprocess.check_call(cmd)

In [ ]:
# 7. Evaluate: base vs trained
!python eval.py --env-url $ENV_URL --episodes-per-task 3

## Training tips

- Start with persona=`senior` for consistent shaping; switch to `principal` later for stricter critiques.
- Enable adversarial rollouts (`--use-adversarial`) only after the curriculum tier reaches `expert` (≈episode 50+).
- Watch for reward plateaus: our shaped reward + phase bonus usually produces +3 to +8 on solved episodes. If you see a flat ~1.0, the model is getting judge score + step bonuses without ever applying the correct mitigation.
- Checkpoints are saved every `save_steps` to `./checkpoints/grpo` and (if `--hub-repo` is set) pushed to the Hub.